# Test Results Analysis — `testimages` Evaluation

Analysis of how each trained model performed on the held-out `testimages` folder.

**Source:** `../results/model_predictions.csv`  
**Classes:** c0 (safe driving) → c9 (talking to passenger)  
**Models evaluated:** Logistic Regression, Decision Tree, Naive Bayes, Random Forest, SVM (PCA), CNN, MobileNetV2 Transfer, LSTM, Transformer, RL Q-Network

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

df = pd.read_csv("../results/model_predictions.csv")

CLASS_NAMES = [f"c{i}" for i in range(10)]
CLASS_LABELS = {
    "c0": "c0 safe driving",
    "c1": "c1 texting-R",
    "c2": "c2 phone-R",
    "c3": "c3 texting-L",
    "c4": "c4 phone-L",
    "c5": "c5 radio",
    "c6": "c6 drinking",
    "c7": "c7 reach-back",
    "c8": "c8 hair/makeup",
    "c9": "c9 talking",
}

MODEL_COLS = [
    "Logistic_Regression",
    "Decision_Tree",
    "Naive_Bayes",
    "Random_Forest",
    "SVM_PCA",
    "CNN",
    "MobileNetV2_Transfer",
    "LSTM",
    "Transformer",
    "RL_Q_Network",
]

MODEL_LABELS = {
    "Logistic_Regression":   "Logistic Reg.",
    "Decision_Tree":         "Decision Tree",
    "Naive_Bayes":           "Naive Bayes",
    "Random_Forest":         "Random Forest",
    "SVM_PCA":               "SVM (PCA)",
    "CNN":                   "CNN",
    "MobileNetV2_Transfer":  "MobileNetV2",
    "LSTM":                  "LSTM",
    "Transformer":           "Transformer",
    "RL_Q_Network":          "RL Q-Net",
}

y_true = df["expected"].values
print(f"Total images: {len(df)}")
print(f"Class distribution:\n{df['expected'].value_counts().sort_index()}")

## 1 — Overall Model Accuracy

In [ ]:
accuracies = {col: float(np.mean(df[col].values == y_true)) for col in MODEL_COLS}
acc_series = pd.Series(accuracies).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ["#2ecc71" if v >= 0.95 else "#3498db" if v >= 0.75 else "#e74c3c" for v in acc_series.values]
bars = ax.barh([MODEL_LABELS[m] for m in acc_series.index], acc_series.values * 100, color=colors)

for bar, val in zip(bars, acc_series.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val*100:.1f}%", va="center", fontsize=9)

ax.set_xlim(0, 115)
ax.set_xlabel("Accuracy (%)")
ax.set_title("Overall Accuracy on Testimages (all 10 classes)")
ax.axvline(80, color="gray", linestyle="--", linewidth=0.8, label="80% reference")
ax.legend(fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAccuracy table:")
for model, acc in acc_series.items():
    correct = int(acc * len(df))
    print(f"  {MODEL_LABELS[model]:<20} {correct:3d}/{len(df)}  ({acc*100:.1f}%)")

## 2 — Per-Class Accuracy Heatmap (all models vs all classes)

In [ ]:
# Build per-class accuracy matrix: rows=models, cols=classes
per_class_acc = pd.DataFrame(index=MODEL_COLS, columns=CLASS_NAMES, dtype=float)

for cls in CLASS_NAMES:
    mask = y_true == cls
    for model in MODEL_COLS:
        if mask.sum() == 0:
            per_class_acc.loc[model, cls] = np.nan
        else:
            per_class_acc.loc[model, cls] = float(np.mean(df[model].values[mask] == y_true[mask]))

per_class_acc.index = [MODEL_LABELS[m] for m in MODEL_COLS]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    per_class_acc.astype(float) * 100,
    annot=True, fmt=".0f", cmap="RdYlGn",
    vmin=0, vmax=100,
    linewidths=0.4, linecolor="white",
    xticklabels=[CLASS_LABELS[c] for c in CLASS_NAMES],
    ax=ax,
    cbar_kws={"label": "Accuracy (%)"},
)
ax.set_title("Per-Class Accuracy per Model (%)", fontsize=12)
ax.set_xlabel("Class")
ax.set_ylabel("Model")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 3 — Confusion Matrices (one per model)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for idx, model in enumerate(MODEL_COLS):
    y_pred = df[model].values
    cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)

    # Normalize row-wise so each row = recall for that class
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm.astype(float) / row_sums

    ax = axes[idx]
    sns.heatmap(
        cm_norm * 100, ax=ax,
        annot=True, fmt=".0f",
        cmap="Blues", vmin=0, vmax=100,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.3, linecolor="white",
        cbar=False,
    )
    acc = accuracies[model]
    ax.set_title(f"{MODEL_LABELS[model]}\nAcc={acc*100:.1f}%", fontsize=9)
    ax.set_xlabel("Predicted", fontsize=7)
    ax.set_ylabel("Actual", fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle("Normalized Confusion Matrices (row = recall %)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4 — Per-Class Sample Count vs Accuracy (bubble chart)

In [ ]:
class_counts = pd.Series(y_true).value_counts().sort_index()

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for idx, model in enumerate(MODEL_COLS):
    ax = axes[idx]
    y_pred = df[model].values

    cls_acc = []
    for cls in CLASS_NAMES:
        mask = y_true == cls
        if mask.sum() > 0:
            cls_acc.append(float(np.mean(y_pred[mask] == y_true[mask])) * 100)
        else:
            cls_acc.append(0.0)

    counts = [class_counts.get(c, 0) for c in CLASS_NAMES]
    colors_scatter = ["#2ecc71" if a >= 80 else "#e74c3c" for a in cls_acc]

    sc = ax.scatter(CLASS_NAMES, cls_acc,
                    s=[c * 8 for c in counts],
                    c=colors_scatter, alpha=0.8, edgecolors="white", linewidths=0.5)
    ax.set_ylim(-5, 110)
    ax.set_ylabel("Acc (%)", fontsize=7)
    ax.set_title(MODEL_LABELS[model], fontsize=9)
    ax.axhline(100, color="gray", linestyle="--", linewidth=0.6)
    ax.tick_params(labelsize=7)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Per-Class Accuracy per Model  (bubble size ∝ sample count)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5 — Misclassification Analysis

Which classes each model confuses most, and which images were hardest (misclassified by many models).

In [ ]:
# How many models got each image wrong
df["num_wrong"] = sum((df[m] != df["expected"]).astype(int) for m in MODEL_COLS)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: distribution of "how many models failed this image"
counts_wrong = df["num_wrong"].value_counts().sort_index()
axes[0].bar(counts_wrong.index, counts_wrong.values, color="#3498db", edgecolor="white")
axes[0].set_xlabel("Number of models that misclassified the image")
axes[0].set_ylabel("Number of images")
axes[0].set_title("Image Difficulty Distribution")
axes[0].grid(axis="y", alpha=0.3)
for x, y in zip(counts_wrong.index, counts_wrong.values):
    axes[0].text(x, y + 0.3, str(y), ha="center", fontsize=8)

# Right: avg difficulty score per class
avg_wrong = df.groupby("expected")["num_wrong"].mean().reindex(CLASS_NAMES)
bar_colors = ["#e74c3c" if v > 2 else "#f39c12" if v > 0.5 else "#2ecc71" for v in avg_wrong]
axes[1].bar([CLASS_LABELS[c] for c in CLASS_NAMES], avg_wrong.values, color=bar_colors, edgecolor="white")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Avg. models wrong")
axes[1].set_title("Average Model Failures per Class")
axes[1].set_xticklabels([CLASS_LABELS[c] for c in CLASS_NAMES], rotation=35, ha="right")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 10 hardest images (most models wrong):")
hard = df[["image", "expected", "num_wrong"]].sort_values("num_wrong", ascending=False).head(10)
print(hard.to_string(index=False))

## 6 — Error Type Breakdown (what each model misclassifies as)

In [ ]:
# For each model collect (true_label, predicted_label) pairs where prediction is wrong
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for idx, model in enumerate(MODEL_COLS):
    ax = axes[idx]
    wrong = df[df[model] != df["expected"]]

    if wrong.empty:
        ax.text(0.5, 0.5, "No errors!", ha="center", va="center", fontsize=12)
        ax.set_title(MODEL_LABELS[model], fontsize=9)
        continue

    # Count (true → predicted) pairs
    pair_counts = wrong.groupby(["expected", model]).size().reset_index(name="count")
    pair_counts.columns = ["true", "predicted", "count"]
    pair_counts["pair"] = pair_counts["true"] + "→" + pair_counts["predicted"]
    pair_counts = pair_counts.sort_values("count", ascending=True).tail(10)

    colors_bar = [f"C{CLASS_NAMES.index(r)}" for r in pair_counts["predicted"]]
    ax.barh(pair_counts["pair"], pair_counts["count"], color=colors_bar, alpha=0.8)
    ax.set_title(f"{MODEL_LABELS[model]}  ({len(wrong)} errors)", fontsize=9)
    ax.set_xlabel("Count", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Top Misclassification Pairs per Model (true → predicted)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7 — Model Agreement Heatmap

How often do pairs of models agree on the same prediction (regardless of correctness)?

In [ ]:
n = len(MODEL_COLS)
agree_matrix = np.zeros((n, n))

for i, m1 in enumerate(MODEL_COLS):
    for j, m2 in enumerate(MODEL_COLS):
        agree_matrix[i, j] = float(np.mean(df[m1].values == df[m2].values)) * 100

labels = [MODEL_LABELS[m] for m in MODEL_COLS]
agree_df = pd.DataFrame(agree_matrix, index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(n, dtype=bool)   # hide diagonal (always 100%)
sns.heatmap(
    agree_df, annot=True, fmt=".0f",
    cmap="YlOrRd", vmin=60, vmax=100,
    mask=mask,
    linewidths=0.4, linecolor="white",
    ax=ax,
    cbar_kws={"label": "Agreement (%)"},
)
# Fill diagonal manually so it reads 100
for i in range(n):
    ax.text(i + 0.5, i + 0.5, "100", ha="center", va="center",
            fontsize=9, color="black", weight="bold")

ax.set_title("Pairwise Model Agreement (% same prediction)", fontsize=12)
plt.tight_layout()
plt.show()

## 8 — Full Classification Report (per model)

In [ ]:
for model in MODEL_COLS:
    print("=" * 60)
    print(f" {MODEL_LABELS[model]}")
    print("=" * 60)
    print(classification_report(
        y_true, df[model].values,
        labels=CLASS_NAMES,
        target_names=[CLASS_LABELS[c] for c in CLASS_NAMES],
        zero_division=0,
    ))
    print()

## 9 — Summary Table (Accuracy, Macro F1, Macro Precision, Macro Recall)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

summary_rows = []
for model in MODEL_COLS:
    y_pred = df[model].values
    summary_rows.append({
        "Model": MODEL_LABELS[model],
        "Accuracy": round(float(np.mean(y_pred == y_true)) * 100, 1),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro", zero_division=0) * 100, 1),
        "Macro Prec.": round(precision_score(y_true, y_pred, average="macro", zero_division=0) * 100, 1),
        "Macro Recall": round(recall_score(y_true, y_pred, average="macro", zero_division=0) * 100, 1),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
display(summary_df.style
    .background_gradient(subset=["Accuracy", "Macro F1", "Macro Prec.", "Macro Recall"],
                         cmap="RdYlGn", vmin=50, vmax=100)
    .format("{:.1f}", subset=["Accuracy", "Macro F1", "Macro Prec.", "Macro Recall"])
    .set_caption("Model Performance Summary on Testimages (%)")
)

# Also render as bar chart
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(summary_df))
w = 0.2
for i, metric in enumerate(["Accuracy", "Macro F1", "Macro Prec.", "Macro Recall"]):
    ax.bar(x + i * w, summary_df[metric], width=w, label=metric)
ax.set_xticks(x + 1.5 * w)
ax.set_xticklabels(summary_df["Model"], rotation=30, ha="right")
ax.set_ylabel("%")
ax.set_ylim(0, 115)
ax.set_title("Model Summary Metrics on Testimages")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()